# Initial Setup

This should take about 2 minutes on Colab.

In [1]:
!apt-get install poppler-utils tesseract-ocr libmagic-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  libmagic-dev poppler-utils
0 upgraded, 2 newly installed, 0 to remove and 2 not upgraded.
Need to get 291 kB of archives.
After this operation, 1,086 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libmagic-dev amd64 1:5.41-3ubuntu0.1 [105 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 291 kB in 2s (170 kB/s)
Selecting previously unselected package libmagic-dev:amd64.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../libmagic-dev_1%3a5.41-3ubuntu0.1_amd64.deb ...
Unpacking libmagic-dev:amd64 (1:5.41-3ubuntu0.1) ...
Selecting previously unselected package poppler-utils.
Preparing to unpack .../poppler-u

In [2]:
%pip install -Uq "unstructured[all-docs]"
%pip install -Uq langchain_chroma
%pip install -Uq langchain langchain-community langchain_google_genai
%pip install -Uq python_dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 43.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 538.2/538.2 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 11.9 MB/s eta 0:00:00
   ━

In [4]:
import json

# Unstructured for document parsing
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

# LangChain components
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from dotenv import load_dotenv

load_dotenv()

import os
import shutil

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
!cp "/content/drive/MyDrive/RAG Project/attention-is-all-you-need.pdf" "attention-is-all-you-need.pdf"

In [26]:
from google.colab import userdata
gemini_api_key = userdata.get('gemini_api_key') # gemini_api_key should be defined in Colab Secrets (key icon on left sidebar)

# You can create a Gemini API key here if you don't have one: https://aistudio.google.com/app/api-keys
# You can check your usage here: https://aistudio.google.com/app/usage?timeRange=last-28-days


# Ingestion Pipeline

You only need to run this part once per document.

This should take about 5 minutes on Colab for the "Attention is All You Need" PDF file. The required time varies depending on the file size.

In [7]:
def partition_document(file_path: str):
    """Extract elements from PDF using unstructured"""
    print(f"Partitioning document: {file_path}")

    elements = partition_pdf(
        filename=file_path,
        strategy="hi_res",
        infer_table_structure=True,
        extract_image_block_types=["Image"],
        extract_image_block_to_payload=True
    )

    print(f"Extracted {len(elements)} elements")
    return elements

In [8]:
def create_chunks_by_title(elements):
    """Create chunks using title-based strategy"""
    print("Chunking...")

    chunks = chunk_by_title(
        elements,
        max_characters=3000, # Max 3000 characters per chunk
        new_after_n_chars=2400, # Try to start a new chunk after 2400 characters
        combine_text_under_n_chars=500 # Merge small chunks under 500 chars with neighbours
    )

    print(f"Created {len(chunks)} chunks")
    return chunks

In [11]:
def separate_content_types(chunk):
    """Analyze what types of content are in a chunk"""
    content_data = {
        'text': chunk.text,
        'tables': [],
        'images': [],
        'types': ['text']
    }

    # Check for tables and images in original elements
    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__

            # Handle tables
            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)

            # Handle images
            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)

    content_data['types'] = list(set(content_data['types']))
    return content_data

def create_ai_enhanced_summary(text, tables, images):
    """Create AI-enhanced summary for mixed content"""

    try:
        llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview", timeout = 30, api_key = gemini_api_key)

        prompt_text = f"""You are creating a searchable description for document content retrieval.

        CONTENT TO ANALYZE:
        TEXT CONTENT:
        {text}

        """

        # Add tables if present
        if tables:
            prompt_text += "TABLES:\n"
            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n{table}\n\n"

                prompt_text += """
                YOUR TASK:
                Generate a comprehensive, searchable description that covers:

                1. Key facts, numbers, and data points from text and tables
                2. Main topics and concepts discussed
                3. Questions this content could answer
                4. Visual content analysis (charts, diagrams, patterns in images)
                5. Alternative search terms users might use

                Make it detailed and searchable - prioritize findability over brevity.

                SEARCHABLE DESCRIPTION:"""

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]

        # Add images to the message
        for image_base64 in images:
            message_content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
            })

        # Send to AI and get response
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])

        return response.text

    except Exception as e:
        print(f"     AI summary failed: {e}")
        # Fallback to simple summary in case of failure
        summary = f"{text[:300]}..."
        if tables:
            summary += f" [Contains {len(tables)} table(s)]"
        if images:
            summary += f" [Contains {len(images)} image(s)]"
        return summary

def summarise_chunks(chunks):
    """Process all chunks with AI Summaries"""
    print("Processing chunks with AI Summaries...")

    langchain_documents = []
    total_chunks = len(chunks)

    for i, chunk in enumerate(chunks):
        current_chunk = i + 1
        print(f"   Processing chunk {current_chunk}/{total_chunks}")

        # Analyze chunk content
        content_data = separate_content_types(chunk)

        # Debug prints
        print(f"     Types found: {content_data['types']}")
        print(f"     Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")

        # Create AI-enhanced summary if chunk has tables/images
        if content_data['tables'] or content_data['images']:
            print(f"     → Creating AI summary for mixed content...")
            try:
                enhanced_content = create_ai_enhanced_summary(
                    content_data['text'],
                    content_data['tables'],
                    content_data['images']
                )
                print(f"     → AI summary created successfully")
                print(f"     → Enhanced content preview: {enhanced_content[:200]}...")
            except Exception as e:
                print(f"     AI summary failed: {e}")
                enhanced_content = content_data['text']
        else:
            print(f"     → Using raw text (no tables/images)")
            enhanced_content = content_data['text']

        # Create LangChain Document with rich metadata
        doc = Document(
            page_content=enhanced_content,
            metadata={
                "original_content": json.dumps({
                    "raw_text": content_data['text'],
                    "tables_html": content_data['tables'],
                    "images_base64": content_data['images']
                })
            }
        )

        langchain_documents.append(doc)

    print(f"Processed {len(langchain_documents)} chunks")
    return langchain_documents

In [12]:
def create_vector_store(documents, persist_directory="dbv2/chroma_db"):
    """Create and persist ChromaDB vector store"""
    print("Creating embeddings and storing in ChromaDB...")

    # Clear existing directory if it exists
    if os.path.exists(persist_directory):
        print(f"Clearing existing directory: {persist_directory}")
        shutil.rmtree(persist_directory)

    # Recreate the directory
    os.makedirs(persist_directory, exist_ok=True)

    embedding_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", timeout = 30, api_key = gemini_api_key)

    # Create ChromaDB vector store
    print("--- Creating vector store ---")
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory,
        collection_metadata={"hnsw:space": "cosine"}
    )
    print("--- Finished creating vector store ---")

    print(f"Vector store created and saved to {persist_directory}")
    return vectorstore

In [15]:
def run_complete_ingestion_pipeline(pdf_path: str):
    """Run the complete RAG ingestion pipeline"""
    print("Starting RAG Ingestion Pipeline")

    # Step 1: Partition
    elements = partition_document(pdf_path)

    # Step 2: Chunk
    chunks = create_chunks_by_title(elements)

    # Step 3: AI Summarisation
    summarised_chunks = summarise_chunks(chunks)

    # Step 4: Vector Store
    db = create_vector_store(summarised_chunks, persist_directory="dbv2/chroma_db")

    print("Pipeline completed successfully")
    return db

In [14]:
db = run_complete_ingestion_pipeline("attention-is-all-you-need.pdf")

Starting RAG Ingestion Pipeline
Partitioning document: attention-is-all-you-need.pdf


yolox_l0.05.onnx:   0%|          | 0.00/217M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/274 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

Extracted 230 elements
Chunking...
Created 25 chunks
Processing chunks with AI Summaries...
   Processing chunk 1/25
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 2/25
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 3/25
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 4/25
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 5/25
     Types found: ['image', 'text']
     Tables: 0, Images: 1
     → Creating AI summary for mixed content...
     → AI summary created successfully
     → Enhanced content preview: ### Searchable Document Description

**Title:** Transformer Model Architecture Overview

**Summary:**
This document describes the fundamental architecture of the Transformer neural network model. It e...
   Processing chunk 6

# Retrieval Pipeline

This needs to be run for every query. Response time should be near instantaneous, but a timeout has been set for 30 seconds in case there's no response from the LLM for some reason.

In [18]:
 # Store conversation as messages
chat_history = [SystemMessage(content="There is no older conversation.")]

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview", timeout = 30, api_key = gemini_api_key)

In [20]:
def ask_question(user_question):
    print(f"\n--- You asked: {user_question} ---")

    # Step 1: Make the question clear using conversation history
    global chat_history
    if len(chat_history) > 1:
        # Ask AI to make the question standalone
        messages = [
            SystemMessage(content="Given the chat history, rewrite the new question to be standalone and searchable. Just return the rewritten question. If the question is already standalone and needs no additional context, just return the original question itself. In either case, do NOT answer the question itself."),
            *chat_history,
            HumanMessage(content=f"New question: {user_question}"),
            ]

        result = llm.invoke(messages)
        search_question = result.text.strip()
        print(f"Searching for: {search_question}")
    else:
        search_question = user_question

    # Step 2: Find relevant documents
    retriever = db.as_retriever(search_kwargs={"k": 5})
    chunks = retriever.invoke(search_question)

    print(f"Found {len(chunks)} relevant chunks:")
    for i, chunk in enumerate(chunks, 1):
        # Show first 2 lines of each chunk
        lines = chunk.page_content.split("\n")[:2]
        preview = "\n".join(lines)
        print(f"  Chunk {i}: {preview}...")

    # Step 3: Create final prompt
    prompt_text = f"""Based on the following documents, please answer this question: {user_question}

CONTENT TO ANALYZE:
"""

    for i, chunk in enumerate(chunks):
        prompt_text += f"--- Document {i+1} ---\n"

        if "original_content" in chunk.metadata:
            original_data = json.loads(chunk.metadata["original_content"])

            # Add raw text
            raw_text = original_data.get("raw_text", "")
            if raw_text:
                prompt_text += f"TEXT:\n{raw_text}\n\n"

            # Add tables as HTML
            tables_html = original_data.get("tables_html", [])
            if tables_html:
                prompt_text += "TABLES:\n"
                for j, table in enumerate(tables_html):
                    prompt_text += f"Table {j+1}:\n{table}\n\n"

        prompt_text += "\n"

    prompt_text += """
Please provide a clear, comprehensive answer using the text, tables, and images above. If the documents don't contain sufficient information to answer the question, say "I don't have enough information to answer that question based on the provided documents."
However, if it's a trivial question like 'Hello' or 'What is 2 + 2?', you may respond without consulting the documents.

ANSWER:"""

    # Build message content starting with text
    message_content = [{"type": "text", "text": prompt_text}]

    # Add all images from all chunks
    for chunk in chunks:
        if "original_content" in chunk.metadata:
            original_data = json.loads(chunk.metadata["original_content"])
            images_base64 = original_data.get("images_base64", [])

            for image_base64 in images_base64:
                message_content.append({
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
                })

    # Step 4: Get the answer
    messages = [
        SystemMessage(content="You are a helpful assistant that answers questions based on provided documents and conversation history."),
        *chat_history,
        HumanMessage(content=message_content),
        ]

    result = llm.invoke(messages)
    answer = result.text

    # Step 5: Remember this conversation
    chat_history.append(HumanMessage(content=user_question))
    chat_history.append(AIMessage(content=answer))

    print(f"Answer: {answer}")

    # Step 6: Summarise older chat history (after 10 interactions)
    if(len(chat_history) == 21):
        instr = [
            *chat_history[:11],
            HumanMessage("Provide a summary of the human-AI conversation given above that captures the key points. Title it as 'Summary of Older Conversation'"),
        ]
        summary = llm.invoke(instr).text

        chat_history = [SystemMessage(content=summary)] + chat_history[11:]
        print("\nThe older chat history has been summarised.")

# Simple chat loop
def start_chat():
    print("The chatbot has started. Type 'quit' to exit and 'delete' to erase chat history.")

    while True:
        question = input("\nEnter your question: ")

        if question.lower() == "quit":
            print("The chatbot has ended.")
            break

        if question.lower() == "delete":
            global chat_history
            chat_history = [SystemMessage(content="There is no older conversation.")]
            print("The chat history has been deleted.")
            continue

        ask_question(question)

In [27]:
start_chat()

# Sample Questions:
# What are the two main components of the Transformer architecture?
# What is the dimension of each head in this type of architecture?
# How many transformations does the feed-forward network in the Transformer architecture contain?

The chatbot has started. Type 'quit' to exit and 'delete' to erase chat history.

Enter your question: quit
The chatbot has ended.
